# 7. Relevancia de variables por vano y ventana, sobre todo el dataset

Hermano de lote del `06_uiti_vano_explicabilidad_simulador`. El 06 contesta, para el
punado de vanos que hay en pantalla, **que variable y en que valor bajaria a ese vano al
grupo Bajo**. Este contesta lo mismo para **todas** las celdas (vano, ventana) del
dataset, y lo deja en una hoja de calculo que se puede repartir.

**Que produce.**

1. Un **grafico de barras por grupo de criticidad** -- Alto, Medio-Alto y Medio -- con las
   diez variables que en promedio mas bajan el UITI de las bolsas de ese grupo. Contesta
   la pregunta de planeacion: no que mueve a UN vano, sino que mueve al grupo.
2. Un **Excel** con una fila por (vano, ventana): su circuito, su grupo en esa ventana --
   o la etiqueta `sin eventos` cuando no registro ninguno -- y el top 10 de variables. Para
   una celda por encima de Bajo, el top son las que la BAJARIAN; para una que ya esta en
   Bajo, las que la SACARIAN de ahi, que es de lo que depende que se quede.

**Un solo modelo y una sola unidad.** Todo sale del MIL entrenado en
`05_mil_vano_ventana`, que puntua bolsas, y de la geometria KMeans de 01.4, la misma que
pinta los mapas de 04 y 06. Los grupos de esta hoja SON los de esos mapas.

**Solo variables de intervencion y de escenario.** Las refutadas -- como los trafos
afectados en la falla, que se miden despues del evento que el modelo anticipa -- y las de
lectura unica quedan fuera de todo el analisis.

**Requiere los dos artefactos de `05_mil_vano_ventana.ipynb`**:
`data/models/mil_vano_ventana_v1.pt` y `data/derived/bolsas_mil_full.joblib`. Los dos
viven bajo `data/`, que git ignora; si faltan, la celda del modelo falla de inmediato
nombrando el cuaderno que los produce.

Corre de punta a punta sin interaccion: **un minuto** de barrido y el Excel escrito.


In [ ]:
import asyncio
import sys
import time
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from IPython.display import display

# Sube desde el cwd hasta la raiz del repo (marcada por la carpeta src/), igual que 09.
# Se agregan ROOT y ROOT/src -- no solo src/ -- porque ventanas_015.py importa
# `scripts.extract_geometrias_014` (paquete de nivel de repo, igual que en notebook 10).
ROOT = Path.cwd().resolve()
while not (ROOT / 'src').is_dir() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
for _path_a_agregar in (ROOT, ROOT / 'src'):
    if str(_path_a_agregar) not in sys.path:
        sys.path.insert(0, str(_path_a_agregar))

# Un kernel que ya importo estos paquetes se queda con la version VIEJA en `sys.modules`:
# "Run All" sin reiniciar NO vuelve a leer el disco. Un rename en src/ -- por ejemplo
# `SelectorVanos._caja` -> `.caja` -- estalla entonces como AttributeError diez celdas mas
# abajo, con el codigo del disco ya correcto. Se purgan ANTES de importarlos, asi el
# cuaderno corre SIEMPRE contra la fuente actual, con o sin reinicio de kernel. Va aqui y no
# como `importlib.reload`: reload no rehace los objetos ya construidos con la clase vieja,
# y este cuaderno los reconstruye todos de esta celda para abajo.
for _modulo in [m for m in list(sys.modules)
                if m.split('.')[0] in ('chec_impacto', 'chec_local_interpreter', 'scripts')]:
    del sys.modules[_modulo]

from chec_impacto.data import procesar_dataset_completo
from chec_impacto.models.criticality_assignment import (
    CLAVE_ESPACIO_CANONICO,
    GEOMETRIAS_SHA1_ESPERADO,
    cargar_geometria_014,
    verificar_sha1_geometrias,
)
from chec_impacto.data.bags import cargar_bolsas
from chec_impacto.models.mil_persistencia import cargar_modelo_mil
from chec_impacto.training import resolve_training_device
from chec_local_interpreter.costos_items import (
    MAX_REPETICIONES,
    costos_de_intervencion,
    leer_catalogo_costos,
)
from chec_local_interpreter.relevancia_lote import (
    SIN_EVENTOS,
    barrer_todas_las_bolsas,
    guardar_hojas,
    ranking_por_bolsa,
    relevancia_media_por_grupo,
    tabla_vano_ventana,
)
from chec_local_interpreter.mil_simulador_015 import (
    gates_de_bolsas,
    grafo_de_gates,
    plan_hacia_clase_minima,
    relevancia_hacia_uiti_minimo,
    seleccionar_bolsas,
    simular_bolsas,
    trazas_grafo,
    valores_actuales_por_vano,
)
from chec_local_interpreter.simulador_variables import (
    GRUPO_POR_KNOB,
    columnas_panel,
    rotulo_en_barra,
    knobs_bloqueados,
    knobs_simulables,
    tabla_variables_simulables,
)
from chec_local_interpreter.vano_controls import build_knobs, expand_knob_overrides
from chec_local_interpreter.vano_widgets import (
    MAX_VANOS_ANALISIS,
    construir_selector_casillas,
    construir_selector_vanos,
)
from chec_local_interpreter.ventanas_015 import (
    CAMBIOS,
    CAMBIO_EMPEORA,
    CAMBIO_IGUAL,
    CAMBIO_MEJORA,
    bounds_de_fids,
    cajas_por_cambio_de_grupo,
    cajas_seleccion,
    capas_mapa_historico,
    cargar_clases_desde_014,
    centro_y_zoom,
    construir_hist_class_cache,
    construir_mask_cache,
    construir_tabla_vano_ventana,
    construir_ventanas,
    fid_de_punto,
    clases_de_series,
    series_temporal_vanos,
)
from scripts.extract_geometrias_014 import (
    DEFAULT_NOTEBOOK_PATH,
    DEFAULT_OUTPUT_PATH,
    extraer_geometrias_014,
)


In [ ]:
# El espacio de agrupamiento no se elige: viene fijo de `criticality_assignment.py`.
CLAVE_ESPACIO = CLAVE_ESPACIO_CANONICO
VENTANA_CLIMATICA_HORAS = 12
# Misma paleta y mismos nombres que 01.4, 04 y 06: un color tiene que significar lo mismo
# al pasar de un cuaderno a otro, o el lector aprende un codigo por cuaderno.
NOMBRES_GRUPOS = ['Bajo', 'Medio', 'Medio-Alto', 'Alto']
COLORES_GRUPOS = ['rgb(252,187,161)', 'rgb(251,106,74)', 'rgb(203,24,29)', 'rgb(103,0,13)']
PALETA_MODALIDADES = ['#0d9488', '#be185d']
# Cuantos valores se le prueban a cada control. Nueve, como en el 06 y por lo mismo:
# medido, 10 de los 15 controles numericos tienen su mejor valor en el INTERIOR del rango
# para alguna bolsa, asi que los dos extremos son los dos puntos equivocados.
PUNTOS_REJILLA = 9
TOP_VARIABLES = 10
# Colores de los dos grupos de variables en las barras: el color dice de que MITAD de la
# decision viene cada una -- lo que se hace y lo que se anticipa -- y por eso no toca la
# paleta de criticidad, donde un rojo significa un grupo.
COLOR_INTERVENCION = '#0072b2'
COLOR_ESCENARIO = '#e69f00'
DEVICE = resolve_training_device('auto')

In [ ]:
# --- Reutilizacion de la geometria KMeans de 01.4 (design section F) -------
# Falla RAPIDO aqui, antes de procesar el dataset completo (celda siguiente): si 01.4 fue
# editado y sus centroides se movieron, no tiene sentido esperar el procesamiento pesado
# para enterarse. `cargar_clases_desde_014` (celda 7, via hist_class_cache) repite esta
# misma verificacion por cada ventana consultada -- barata, y evita que una geometria
# cacheada quede sin recomprobar dentro de la misma sesion.
GEOMETRIAS_PATH = DEFAULT_OUTPUT_PATH
if not GEOMETRIAS_PATH.exists():
    extraer_geometrias_014(DEFAULT_NOTEBOOK_PATH, GEOMETRIAS_PATH)
_sha1_real, _coincide = verificar_sha1_geometrias(GEOMETRIAS_PATH, esperado=GEOMETRIAS_SHA1_ESPERADO)
assert _coincide, (
    f'La geometria KMeans extraida de 01.4 no coincide con la esperada '
    f'(esperado={GEOMETRIAS_SHA1_ESPERADO}, real={_sha1_real}). 01.4 fue modificado; '
    f'01.5 depende de esa geometria.'
)
# La geometria en si (no solo su sha1): la nube KMeans de la fila 3 tiene que dibujarse en
# el MISMO espacio en que se asignan las clases -- el canonico '2' es (log_x=False,
# log_y=True). Leerlo de la geometria y no fijarlo a mano evita que un cambio de espacio
# deje la nube en ejes que ya no corresponden a las fronteras.
GEOMETRIA_014 = cargar_geometria_014(GEOMETRIAS_PATH, CLAVE_ESPACIO)
print(f'Geometria 01.4 verificada -- sha1 coincide ({_sha1_real[:12]}...) | '
      f'espacio {CLAVE_ESPACIO}: log_x={GEOMETRIA_014.logs[0]}, log_y={GEOMETRIA_014.logs[1]}')

In [ ]:
DATA_PATH = ROOT / 'data' / 'Indicadores_vano_v3.csv'
VARIABLES_SELECCION_PATH = ROOT / 'data' / 'Variables_seleccion.xlsx'
COSTOS_ITEMS_PATH = ROOT / 'data' / 'COSTOS ITEMS CONTRATOS.xlsx'
MODEL_DIR = ROOT / 'data' / 'models'

# Mismo preprocesamiento real usado en entrenamiento (03_mgcecdl_training / 09_simulador):
# sin muestreo ni filtro de UITI, para que context_df quede alineado FILA A FILA con X --
# la clave que permite reusar la MISMA mascara (circuito, ventana) para el mapa historico
# (sin modelo) y, en un PR futuro, para las predicciones del modelo sobre esas mismas filas.
datos = procesar_dataset_completo(
    path_clima=DATA_PATH,
    path_variables_seleccion=VARIABLES_SELECCION_PATH,
    use_sampling=False,
    min_samples_per_codigo=5,
    target='UITI_VANO',
    filtro_uiti_max=None,
    ventana_climatica_horas=VENTANA_CLIMATICA_HORAS,
)

feature_names = list(datos['features'])
X_raw_model = np.asarray(datos['X'], dtype=np.float32)
Xdf = datos['Xdata'].copy().reset_index(drop=True)
context_df = datos['df_original_copy'].copy().reset_index(drop=True)
label_encoders = datos.get('label_encoders', {})
max_values_imputed = datos.get('max_values_imputed', {})

# Ya no se construye el escalador min-max de MGCECDL. El simulador y la importancia de
# variables corren sobre el modelo MIL del cuaderno 05, cuya matriz de instancias es RAW,
# asi que `preparar_splits_estratificados` + `escalar_features_minmax_mgcecdl` -- de lo
# mas caro de esta celda -- no alimentaban ya a nadie.
assert len(context_df) == len(X_raw_model), (
    'context_df y X_raw_model deben quedar alineados fila a fila'
)
print(f'{len(context_df):,} filas | {len(feature_names)} features')

In [ ]:
# --- UN solo modelo: el MIL por bolsas del cuaderno 05 (cierra el SEAM D1) ------------
# El tablero entero -- mapa "Criticidad Simulada", grafo reconstruido e "Importancia
# Variables" -- responde a este modelo y a esta unidad: la BOLSA (vano x ventana), que es
# la unidad en la que 04 define la criticidad. MGCECDL por fila salio del cuaderno: tener
# dos modelos contestando paneles vecinos del mismo tablero significaba que el panel y el
# mapa hablaban de cosas distintas sin que nada en pantalla lo dijera.
# Requiere DOS artefactos que produce el cuaderno 05 y que viven bajo `data/` (ignorado
# por git): el modelo y el cache de bolsas. Falla AQUI, con el nombre del cuaderno que los
# genera, en vez de a los diez minutos en la celda del boton.
RUTA_MODELO_MIL = MODEL_DIR / 'mil_vano_ventana_v1.pt'
RUTA_BOLSAS_MIL = ROOT / 'data' / 'derived' / 'bolsas_mil_full.joblib'
for _ruta in (RUTA_MODELO_MIL, RUTA_BOLSAS_MIL):
    assert _ruta.exists(), (
        f'Falta {_ruta.name}: lo produce 05_mil_vano_ventana.ipynb. Corre ese cuaderno '
        'antes que este.'
    )

BOLSAS = cargar_bolsas(RUTA_BOLSAS_MIL)
X_INST, FEATURES_MIL, BAG_INDEX = BOLSAS['X'], BOLSAS['features'], BOLSAS['bag_index']
# `device='cpu'` a proposito y no DEVICE: una seleccion son decenas de instancias, asi que
# el traslado a GPU/MPS cuesta mas de lo que ahorra, y saca una variable de dtype de en
# medio de un camino interactivo.
MIL = cargar_modelo_mil(RUTA_MODELO_MIL, device='cpu', features_esperadas=FEATURES_MIL)

# La guarda que hace comparables los dos mapas: si el MIL se hubiera entrenado con OTRA
# geometria KMeans, sus clases usarian los mismos 4 colores para significar otra cosa.
for _campo in ('offset', 'scale', 'centroides'):
    assert np.allclose(getattr(MIL.geometria, _campo), getattr(GEOMETRIA_014, _campo)), (
        f'La geometria del modelo MIL difiere de la de 01.4 en {_campo}: sus clases NO son '
        'las del mapa base y no pueden compartir la paleta.'
    )
assert tuple(MIL.geometria.logs) == tuple(GEOMETRIA_014.logs)

# Las 70 primeras features del MIL son exactamente las de MGCECDL (22 estaticas + 48 de
# clima); las 10 restantes son COD_CAUSA y sus indicadores, que no son controles del
# simulador. Por eso el catalogo de knobs de la celda siguiente sirve para los dos.
assert list(FEATURES_MIL[:len(feature_names)]) == list(feature_names), (
    'Las features del MIL ya no empiezan por las de MGCECDL: el catalogo de knobs '
    'apuntaria a columnas equivocadas.'
)
# Los MODOS de variable con los que el modelo agrupa las columnas -- son los mismos que
# usa la fusion FiLM (el clima reescala lo estructural), asi que colorear los nodos del
# grafo por modalidad muestra exactamente la particion que el modelo usa por dentro.
COLUMNAS_MODALIDAD = {m: set(int(i) for i in idx)
                      for m, idx in MIL.model.base.modality_feature_indices.items()}
MODALIDADES_MIL = list(COLUMNAS_MODALIDAD)
assert len(MODALIDADES_MIL) == len(PALETA_MODALIDADES), (
    f'El artefacto trae {len(MODALIDADES_MIL)} modalidades y la figura tiene trazas de '
    f'nodo para {len(PALETA_MODALIDADES)}: agrega la traza que falta antes de seguir.'
)
COLORES_MODALIDAD = dict(zip(MODALIDADES_MIL, PALETA_MODALIDADES))

print(f'MIL cargado -- {len(BAG_INDEX.keys):,} bolsas | {X_INST.shape[0]:,} instancias x '
      f'{len(FEATURES_MIL)} features | geometria identica a 01.4')
print('modos de variable: ' + ' | '.join(
    f'{m} ({len(COLUMNAS_MODALIDAD[m])})' for m in MODALIDADES_MIL))

In [ ]:
# --- Las ventanas, solo para rotular con fechas ----------------------------------------
# La rejilla (vano, ventana) NO se arma desde aqui: las claves de las bolsas ya la traen
# (`CIRCUITO`, `FID_VANO`, `VENTANA`), que es la misma unidad que puntua el modelo. Estas
# ventanas se cargan para poder escribir el periodo al lado de la etiqueta `V1`... `V11`.
VENTANAS = construir_ventanas(context_df['FECHA'])
PERIODO_POR_VENTANA = {v['etiqueta']: v['periodo'] for v in VENTANAS}
ETIQUETAS_VENTANA = [v['etiqueta'] for v in VENTANAS]
print(f'{len(VENTANAS)} ventanas: ' + ' | '.join(
    f'{v["etiqueta"]} {v["periodo"]}' for v in VENTANAS[:3]) + ' ...')

In [ ]:
KNOBS = build_knobs(
    feature_names=feature_names,
    original_feature_df=Xdf,
    label_encoders=label_encoders,
    max_values_imputed=max_values_imputed,
)
print(f'{len(KNOBS)} controles (Knob catalog, PR2a) -- '
      f'{sum(1 for k in KNOBS if k.kind == "categorical")} categoricos, '
      f'{sum(1 for k in KNOBS if k.kind == "numeric")} numericos, '
      f'{sum(1 for k in KNOBS if k.kind == "constant")} constantes')

# Solo las de INTERVENCION y ESCENARIO. Las refutadas y las de lectura unica no entran a
# ningun analisis de relevancia de este cuaderno, por la misma razon por la que el panel
# del 06 no las ofrece: con el catalogo completo, la variable "mas relevante" de un vano
# podria terminar siendo `CNT_TRF` -- los trafos afectados EN LA FALLA, que se miden
# DESPUES del evento que el modelo intenta anticipar. Eso es la flecha del analisis al
# reves, sosteniendo una orden de trabajo que no arregla nada.
KNOBS_PANEL = knobs_simulables(KNOBS)
BLOQUEADOS = knobs_bloqueados(KNOBS)
print(f'{len(KNOBS_PANEL)} controles entran al analisis '
      f'({sum(1 for k in KNOBS_PANEL if GRUPO_POR_KNOB.get(k.id) == "Intervencion")} de '
      f'intervencion, '
      f'{sum(1 for k in KNOBS_PANEL if GRUPO_POR_KNOB.get(k.id) == "Escenario")} de '
      f'escenario)')
print(f'{len(BLOQUEADOS)} quedan fuera: ' + ', '.join(k.id for k in BLOQUEADOS))

## La matematica: como se infiere que variables llevan un vano al grupo Bajo

### El objeto que se mueve

La unidad no es el evento: es la **bolsa** $b$, la celda
$(\text{circuito}, \text{vano}, \text{ventana})$. Es la misma unidad en la que 04 define
la criticidad y en la que 05 entreno el modelo MIL, y por eso los grupos de esta hoja son
exactamente los del mapa.

Cada bolsa aporta sus instancias $I_b$ -- las filas de evento de ese vano en esa ventana --
y dos cantidades que **no se predicen nunca**:

$$n_b=|I_b|\quad(\text{eventos OBSERVADOS}),\qquad x_i\in\mathbb{R}^{80}$$

El modelo produce $\hat u_b=f_\theta(X, b)$, el UITI acumulado predicho de la bolsa, y su
grupo sale de la geometria KMeans de 01.4 sobre
$\zeta_b=\bigl(n_b,\ \log_{10}\hat u_b\bigr)$, estandarizada:

$$\hat k_b=\arg\min_{k\in\{0,1,2,3\}}\lVert \zeta_b-c_k\rVert^2$$

### La meta, y por que no siempre esta al alcance

$n_b$ **no se simula**: es un eje del espacio que define la clase, y moverlo desplazaria
al vano por una dimension que el modelo no predice. Con $n_b$ fijo, el grupo solo depende
de $\hat u$, asi que existe un umbral por debajo del cual la bolsa cae en el grupo Bajo:

$$u^{\star}(n_b)=\max\{\,u>0 \;:\; \arg\min_k\lVert\zeta(n_b,u)-c_k\rVert = 0\,\}$$

Se resuelve por **rejilla** y no por biseccion: nada garantiza que al subir $u$ con $n_b$
fijo se recorran los grupos en orden, y una biseccion asume esa monotonia. Medido sobre la
geometria real, $u^{\star}$ existe en todo el rango de eventos observado pero **se
desploma** al acumularse eventos:

| $n_b$ | 1 | 5 | 10 | 20 | 30 | 46 |
|---|---|---|---|---|---|---|
| $u^{\star}(n_b)$ | 4,41 | 3,93 | 3,37 | 1,15 | 0,114 | 0,0029 |

Un vano con muchos eventos necesita un UITI casi nulo para bajar de grupo. Es una
propiedad del espacio de criticidad, no del simulador, y es la primera cosa que este
cuaderno tiene que decir con claridad: **la meta no cuesta lo mismo para todos**.

### El barrido

Cada control $\kappa$ del panel gobierna un conjunto de columnas $F(\kappa)$ -- una sola
para una variable estatica, las 12 de una familia climatica -- y aporta un conjunto de
candidatos $\mathcal{G}_\kappa$: una rejilla de $G=9$ valores sobre su rango observado si
es numerico, **sus categorias** si es categorico. Aplicarlo es

$$x_{i,j}\;\leftarrow\;\phi_j(v),\qquad \forall\, i,\;\forall\, j\in F(\kappa)$$

con $\phi_j$ la coercion a espacio de modelo. Para cada control se guardan los **dos**
extremos alcanzables de cada bolsa:

$$\hat u^{\downarrow}_{b,\kappa}=\min_{v\in\mathcal{G}_\kappa}\hat u_b\!\left(X^{\kappa\to v}\right),
\qquad
\hat u^{\uparrow}_{b,\kappa}=\max_{v\in\mathcal{G}_\kappa}\hat u_b\!\left(X^{\kappa\to v}\right)$$

Los dos, y no solo el minimo: el minimo contesta *que baja al vano*, el maximo contesta
*de que depende que se quede donde esta*. Cual se usa lo decide el grupo de cada bolsa.

### La metrica, y por que en ordenes de magnitud

$$c_{b,\kappa}=\log_{10}\hat u_b-\log_{10}\hat u^{\downarrow}_{b,\kappa},
\qquad
s_{b,\kappa}=\log_{10}\hat u^{\uparrow}_{b,\kappa}-\log_{10}\hat u_b$$

En unidades de UITI, el ranking de un vano caro seria incomparable con el de uno barato:
$\hat u$ recorre varios ordenes de magnitud entre bolsas. En ordenes de magnitud -- que
ademas es el eje que usa la geometria -- dos valores iguales significan lo mismo en
cualquier grupo, y el promedio por grupo deja de estar dominado por un punado de bolsas
caras.

### La pregunta se invierte segun donde este la bolsa

$$\text{ranking}(b)=
\begin{cases}
\text{las } T \text{ mayores } c_{b,\kappa} & \text{si } \hat k_b\neq 0 \quad(\text{que la BAJA})\\[2pt]
\text{las } T \text{ mayores } s_{b,\kappa} & \text{si } \hat k_b = 0 \quad(\text{de que depende SOSTENERSE})
\end{cases}$$

Una bolsa que ya esta en el grupo Bajo no tiene adonde bajar: ordenarla por caida
alcanzable devuelve $T$ variables que no mueven nada. Lo que lleva informacion es lo
contrario -- sus fragilidades, lo que la sacaria de ahi.

### Reserva por tipo de variable

El top $T=10$ **reserva sitio** para los dos grupos de variables, intervencion y
escenario. Sin la reserva, las cuatro familias climaticas copan la lista y no queda ni una
palanca que una cuadrilla pueda ejecutar; la hoja existe para sostener ordenes de trabajo,
y la mitad de "que pasa si" sin la mitad de "que hago" no sostiene ninguna.

**Solo entran las variables de intervencion y escenario.** Las refutadas y las de lectura
unica quedan fuera de todo el analisis, por la misma razon por la que el panel del 06 no
las ofrece: con el catalogo completo, la variable "mas relevante" de un vano podria
terminar siendo `CNT_TRF` -- los trafos afectados EN LA FALLA, que se miden DESPUES del
evento que el modelo intenta anticipar. Eso es la flecha del analisis al reves.

### Lo que cuesta

Una pasada por candidato, y cada pasada devuelve el $\hat u$ de **todas** las bolsas:

$$\text{pasadas}=1+\sum_{\kappa}|\mathcal{G}_\kappa|$$

Medido: 0,30 s por pasada sobre 288.632 instancias, 197 candidatos, **un minuto** para las
111.233 bolsas. Un bucle por circuito y ventana tardaria dias para obtener exactamente
estos numeros.

### Los limites, dichos de frente

- **Una variable a la vez.** Este cuaderno mide el efecto de mover CADA control por
  separado. Medido en 06 sobre 59 bolsas: en Medio, 20 de 33 alcanzan el grupo Bajo con
  una sola variable; en Medio-Alto, 0 de 18; en Alto, 0 de 8. Arriba de Medio hace falta
  una **combinacion**, y el plan combinado vive en el tablero del 06, no aqui: una hoja
  con un plan por bolsa para 111 mil bolsas cuesta cuatro rondas de barrido completo y
  produce una columna que nadie lee de a 301 mil filas.
- **El optimo es sobre la rejilla**, no sobre el continuo. Con $G=9$ el valor exacto puede
  caer entre dos puntos; lo que se reporta es el mejor de los nueve.
- **Es el modelo, no el mundo.** $\hat u^{\downarrow}$ es lo que el MIL predice si esa
  variable tomara ese valor, con todo lo demas igual. No es una promesa de que la obra
  produzca ese UITI.


In [ ]:
# --- El barrido completo: una pasada por candidato, TODAS las bolsas a la vez ----------
# Aqui esta toda la aritmetica del cuaderno. Cada pasada del modelo ya devuelve un u-hat
# POR BOLSA, asi que barrer las 111 mil bolsas cuesta las MISMAS pasadas que barrer cinco.
# Un bucle por circuito y ventana tardaria dias para obtener exactamente estos numeros.
_t0 = time.perf_counter()
BARRIDO = barrer_todas_las_bolsas(
    MIL, X_INST, instance_bag=BAG_INDEX.instance_bag,
    feature_names=FEATURES_MIL, knobs=KNOBS_PANEL, puntos=PUNTOS_REJILLA,
    label_encoders=label_encoders, max_values_imputed=max_values_imputed,
)
N_OBS = np.asarray(BAG_INDEX.counts, dtype=float)
RANKING = ranking_por_bolsa(BARRIDO, n_obs=N_OBS, geometria=MIL.geometria,
                            top=TOP_VARIABLES, grupos=GRUPO_POR_KNOB)
CLASES = RANKING['clases']
_candidatos = sum(len(c) for c in BARRIDO.candidatos)
print(f'{len(BARRIDO.u_base):,} bolsas x {len(BARRIDO.labels)} controles '
      f'({_candidatos} candidatos) en {time.perf_counter() - _t0:.0f} s')
for _k in range(4):
    _n = int((CLASES == _k).sum())
    print(f'  {NOMBRES_GRUPOS[_k]:<12} {_n:>8,} bolsas ({100 * _n / len(CLASES):.1f}%)')

In [ ]:
# --- Barras: que mueve a cada GRUPO, no a un vano suelto -------------------------------
# Con 111 mil bolsas, el promedio por grupo es lo unico que se lee de un vistazo. Se
# promedia la caida en ORDENES DE MAGNITUD y no en unidades de UITI: un punado de bolsas
# caras se llevaria el promedio entero y el grafico diria mas de ellas que del grupo.
MEDIAS = relevancia_media_por_grupo(BARRIDO, clases=CLASES, n_clases=4)

# Solo los tres grupos que TIENEN adonde bajar. El grupo Bajo no aparece: su pregunta es
# la contraria -- de que depende que se quede -- y mezclarla en el mismo eje presentaria
# dos cosas distintas como comparables.
GRUPOS_CON_CAMINO = [3, 2, 1]
fig_grupos = make_subplots(
    rows=1, cols=len(GRUPOS_CON_CAMINO), shared_yaxes=False,
    subplot_titles=[f'{NOMBRES_GRUPOS[k]} ({int((CLASES == k).sum()):,} bolsas)'
                    for k in GRUPOS_CON_CAMINO],
    horizontal_spacing=0.12,
)
_id_por_label = {k.label: k.id for k in KNOBS_PANEL}
for _col, _clase in enumerate(GRUPOS_CON_CAMINO, start=1):
    _medias = MEDIAS.get(_clase, {})
    if not _medias:
        continue
    _orden = sorted(_medias.items(), key=lambda kv: kv[1])[-TOP_VARIABLES:]
    _labels = [lab for lab, _v in _orden]
    _grupos = [GRUPO_POR_KNOB.get(_id_por_label.get(lab, ''), '') for lab in _labels]
    fig_grupos.add_trace(go.Bar(
        y=_labels, x=[v for _lab, v in _orden], orientation='h', showlegend=False,
        marker=dict(color=[COLOR_INTERVENCION if g == 'Intervencion' else COLOR_ESCENARIO
                           for g in _grupos]),
        hovertext=[f'<b>{lab}</b><br>{g}<br>Caida media: {v:.3f} ordenes de magnitud'
                   for lab, g, (_l, v) in zip(_labels, _grupos, _orden)],
        hoverinfo='text',
    ), row=1, col=_col)
    fig_grupos.update_xaxes(title_text='Caida media de UITI (ordenes de magnitud)',
                            row=1, col=_col)
    fig_grupos.update_yaxes(tickfont=dict(size=9), row=1, col=_col)

# La leyenda se arma con dos trazas vacias: el color de las barras viaja como arreglo y
# una traza por grupo partiria el orden del ranking, que es lo unico que la barra mide.
for _nombre, _color in (('Intervencion', COLOR_INTERVENCION), ('Escenario', COLOR_ESCENARIO)):
    fig_grupos.add_trace(go.Bar(y=[None], x=[None], name=_nombre, orientation='h',
                                marker=dict(color=_color), showlegend=True), row=1, col=1)
fig_grupos.update_layout(
    title=dict(text='Que baja el UITI de cada grupo de criticidad'),
    height=520, autosize=True, template='plotly_white', bargap=0.25,
    margin=dict(l=10, r=14, t=90, b=60),
    legend=dict(orientation='h', x=0.5, xanchor='center', y=-0.16, yanchor='top'),
)
fig_grupos.show()

In [ ]:
# --- La hoja: una fila por (vano, ventana) ---------------------------------------------
# La rejilla es COMPLETA a proposito -- todo vano contra todas las ventanas -- aunque solo
# un tercio de las celdas tenga eventos. Un vano al que le faltan ventanas se lee como que
# no existio en ellas, cuando lo que paso es que no registro eventos.
_t0 = time.perf_counter()
HOJA = tabla_vano_ventana(
    claves=BAG_INDEX.keys, ventanas=ETIQUETAS_VENTANA, clases=CLASES,
    nombres_clase=NOMBRES_GRUPOS, ranking=RANKING, top=TOP_VARIABLES,
)
HOJA.insert(3, 'PERIODO', HOJA['VENTANA'].map(PERIODO_POR_VENTANA).fillna(''))
print(f'{len(HOJA):,} filas en {time.perf_counter() - _t0:.0f} s | '
      f'con eventos: {int((HOJA["GRUPO"] != SIN_EVENTOS).sum()):,} | '
      f'sin eventos: {int((HOJA["GRUPO"] == SIN_EVENTOS).sum()):,}')
HOJA.head(12)

In [ ]:
# --- Guardado ---------------------------------------------------------------------------
# Se escribe por FILAS con `guardar_hojas` y no con `DataFrame.to_excel`. No es una
# preferencia: `to_excel` recorre el DataFrame por COLUMNAS, y el modo `constant_memory`
# de xlsxwriter -- el unico que evita armar 4,8 millones de celdas en RAM -- descarta una
# fila en cuanto el cursor pasa a la siguiente. Juntos producen un archivo que abre sin
# error y trae SOLO la primera columna. Se detecto asi: 1,5 MB y una sola celda con grupo
# en 301 mil filas.
RUTA_EXCEL = ROOT / 'reports' / 'relevancia_variables_por_vano_ventana.xlsx'
RUTA_EXCEL.parent.mkdir(parents=True, exist_ok=True)
_t0 = time.perf_counter()
# Una segunda hoja con el promedio por grupo: es el resumen que sostiene las barras, y sin
# el, quien abra el Excel tiene que reconstruirlo a mano desde 301 mil filas.
RESUMEN = pd.DataFrame([
    {'GRUPO': NOMBRES_GRUPOS[_k], 'VARIABLE': _lab,
     'TIPO': GRUPO_POR_KNOB.get(_id_por_label.get(_lab, ''), ''),
     'CAIDA_MEDIA_ORDENES_DE_MAGNITUD': _v}
    for _k in GRUPOS_CON_CAMINO
    for _lab, _v in sorted(MEDIAS.get(_k, {}).items(), key=lambda kv: -kv[1])
])
guardar_hojas({'vano_ventana': HOJA, 'promedio_por_grupo': RESUMEN}, RUTA_EXCEL)
print(f'{RUTA_EXCEL.relative_to(ROOT)} | {RUTA_EXCEL.stat().st_size / 1e6:.1f} MB | '
      f'{time.perf_counter() - _t0:.0f} s')

## Como leer la hoja

**Una fila por (vano, ventana), y la rejilla es completa.** Solo un tercio de las celdas
tiene eventos; las demas llevan `sin eventos` en `GRUPO` y el top vacio. No es un hueco
que rellenar: sin celda no hay bolsa, sin bolsa no hay prediccion y sin prediccion no hay
ranking. Y `sin eventos` **no es** el grupo Bajo -- es la ausencia del dato.

**`LECTURA_DEL_TOP` dice que pregunta contesta cada fila.** Las dos direcciones no
significan lo mismo, y sin esa columna la hoja se leeria como si el top de un vano en Bajo
fuera "como bajarlo mas". Para una celda por encima de Bajo el top responde *que la
bajaria*; para una que ya esta en Bajo, *de que depende que se quede*.

**Las barras promedian por grupo, no por vano.** Con 111 mil bolsas es lo unico que se lee
de un vistazo, y se promedia en ordenes de magnitud para que un punado de bolsas caras no
se lleve el promedio entero. El color separa las dos mitades de la decision: azul lo que
se hace -- intervencion -- y naranja lo que se anticipa -- escenario.

**El grupo Bajo no tiene barras.** Su pregunta es la contraria y ponerla en el mismo eje
presentaria dos cosas distintas como comparables. Su ranking si esta en la hoja, fila a
fila, con su propia lectura.

**Lo que este cuaderno NO hace.** Mide una variable a la vez. Medido sobre 59 bolsas: en
Medio, 20 de 33 alcanzan el grupo Bajo con una sola variable; en Medio-Alto, 0 de 18; en
Alto, 0 de 8. Arriba de Medio hace falta una combinacion, y el plan combinado vive en el
tablero del 06 -- ahi se elige el vano y se ve la receta completa. Aqui seria una columna
que nadie lee repetida 301 mil veces, y cuatro rondas mas de barrido completo.
